# Silver Layer
- Standardize timestamp format
- Round off decimals


In [1]:
import duckdb
con = duckdb.connect('../olist.duckdb')


In [2]:

# --- Closed deals
con.execute("""
    CREATE OR REPLACE TABLE silver_closed_deals AS
    SELECT
        mql_id,
        seller_id,
        sdr_id,
        sr_id,

        --- format to datetime
        strptime(won_date, '%d/%m/%Y %H:%M') AS won_date,

        business_segment,
        lead_type,
        lead_behaviour_profile,
        has_company,
        has_gtin,

        -- Reformat converted dates back to standard string 
        CASE
            WHEN TRY_CAST(average_stock AS DATE) IS NOT NULL 
                THEN strftime(CAST(average_stock AS DATE), '%d-%m') 
            ELSE CAST(average_stock AS VARCHAR)
        END AS average_stock,
        
        business_type,
        declared_product_catalog_size,
        declared_monthly_revenue

    FROM olist_closed_deals_dataset
""")


In [3]:

# --- Marketing qualified leads 
con.execute("""
    CREATE OR REPLACE TABLE silver_leads AS
    SELECT *
    FROM olist_marketing_qualified_leads_dataset
""")


In [4]:

# --- Orders 
con.execute("""
    CREATE OR REPLACE TABLE silver_orders AS
    SELECT
        order_id,
        customer_id,
        order_status,

        --- format timestamp
        CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
        CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
        CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
        CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
        CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
    FROM olist_orders_dataset
""")


In [5]:

# --- Order reviews
con.execute("""
    CREATE OR REPLACE TABLE silver_order_reviews AS
    SELECT
        review_id,
        order_id,
        review_score,
        review_comment_title,
        review_comment_message,

        --format timestamp
        CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
        CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp
    FROM olist_order_reviews_dataset
""")


In [6]:

# --- Order items
con.execute("""
    CREATE OR REPLACE TABLE silver_order_items AS
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,

        --- format timestamp
        CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,

        --- standardize decimal
        ROUND(price, 2) AS price,
        ROUND(freight_value, 2) AS freight_value
    FROM olist_order_items_dataset
""")


In [7]:
con.execute("""
    CREATE OR REPLACE TABLE silver_payments AS
    SELECT
        order_id,
        payment_sequential,
        payment_type,
        payment_installments,

        --- standardize decimal
        ROUND(payment_value, 2) AS payment_value

    FROM olist_order_payments_dataset
""")

In [8]:

con.execute("""
    CREATE OR REPLACE TABLE silver_customers AS
    SELECT *
    FROM olist_customers_dataset
""")

con.execute("""
    CREATE OR REPLACE TABLE silver_products AS
    SELECT *
    FROM olist_products_dataset
""")

con.execute("""
    CREATE OR REPLACE TABLE silver_sellers AS
    SELECT *
    FROM olist_sellers_dataset
""")

con.execute("""
    CREATE OR REPLACE TABLE silver_geolocation AS
    SELECT *
    FROM olist_geolocation_dataset
""")

con.execute("""
    CREATE OR REPLACE TABLE silver_translation AS
    SELECT *
    FROM product_category_name_translation
""")



print("Silver tables created.")
con.close()

Silver tables created.
